In [42]:
import pandas as pd
import numpy as np
from pathlib import Path

In [43]:
#Load dataset 
processed_dir = Path("D:\Erdos\Data Science Bootcamp\summer26-diabetes-risk\early_diabetes_screening\data\processed")
X_pred_selected = pd.read_csv(processed_dir / "X_pred_selected.csv")
print(X_pred_selected.shape)
X_pred_selected.head()

(9232, 23)


,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPQ020,BPQ080,SMQ020,...,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910,avg_systolic_bp,avg_diastolic_bp
0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,2.0,1.0,2.0,...,1.0,10.0,1.0,3.0,7.0,0.0,0.0,5.0,99.000000,54.333333
1,21.0,2.0,2.0,4.0,5.00,NaN,NaN,2.0,2.0,2.0,...,NaN,NaN,NaN,1.0,4.0,0.0,0.0,0.0,NaN,NaN
2,49.0,1.0,3.0,2.0,NaN,29.7,120.4,2.0,1.0,1.0,...,1.0,0.0,NaN,3.0,2.0,2.0,0.0,0.0,107.000000,67.000000
3,36.0,1.0,3.0,4.0,0.83,21.9,86.8,2.0,2.0,1.0,...,1.0,0.0,NaN,4.0,2.0,2.0,0.0,7.0,113.666667,67.333333
4,68.0,1.0,7.0,4.0,1.20,30.2,109.6,1.0,1.0,2.0,...,1.0,4.0,2.0,2.0,0.0,NaN,0.0,0.0,134.000000,70.000000


In [44]:
X_eng=X_pred_selected.copy()

In [45]:
# Pulse pressure = average systolic BP - average diastolic BP

if "avg_systolic_bp" in X_eng.columns and "avg_diastolic_bp" in X_eng.columns:
    X_eng["pulse_pressure"] = X_eng["avg_systolic_bp"] - X_eng["avg_diastolic_bp"]

X_eng[["avg_systolic_bp", "avg_diastolic_bp", "pulse_pressure"]].head()

,avg_systolic_bp,avg_diastolic_bp,pulse_pressure
0,99.000000,54.333333,44.666667
1,NaN,NaN,NaN
2,107.000000,67.000000,40.000000
3,113.666667,67.333333,46.333333
4,134.000000,70.000000,64.000000


In [46]:
# Obesity indicator
# obese = 1 if BMI >= 30
# obese = 0 if BMI < 30
# obese = NaN if BMI is missing

if "BMXBMI" in X_eng.columns:
    X_eng["obese"] = np.where(
        X_eng["BMXBMI"].isna(),
        np.nan,
        np.where(X_eng["BMXBMI"] >= 30, 1, 0)
    )

X_eng[["BMXBMI", "obese"]].head()

,BMXBMI,obese
0,37.8,1.0
1,NaN,NaN
2,29.7,0.0
3,21.9,0.0
4,30.2,1.0


In [47]:
# BMI category

if "BMXBMI" in X_eng.columns:
    X_eng["bmi_category"] = pd.cut(
        X_eng["BMXBMI"],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=["underweight", "normal", "overweight", "obese"]
    )

X_eng[["BMXBMI", "bmi_category"]].head()

,BMXBMI,bmi_category
0,37.8,obese
1,NaN,NaN
2,29.7,overweight
3,21.9,normal
4,30.2,obese


In [48]:
# High blood pressure indicator
# high_bp_exam = 1 if systolic >= 130 or diastolic >= 80
# high_bp_exam = 0 if both values are known and below threshold
# high_bp_exam = NaN if either BP value is missing

if "avg_systolic_bp" in X_eng.columns and "avg_diastolic_bp" in X_eng.columns:
    X_eng["high_bp_exam"] = np.where(
        X_eng["avg_systolic_bp"].isna() | X_eng["avg_diastolic_bp"].isna(),
        np.nan,
        np.where(
            (X_eng["avg_systolic_bp"] >= 130) |
            (X_eng["avg_diastolic_bp"] >= 80),
            1,
            0
        )
    )

X_eng[["avg_systolic_bp", "avg_diastolic_bp", "high_bp_exam"]].head()

,avg_systolic_bp,avg_diastolic_bp,high_bp_exam
0,99.000000,54.333333,0.0
1,NaN,NaN,NaN
2,107.000000,67.000000,0.0
3,113.666667,67.333333,0.0
4,134.000000,70.000000,1.0


In [49]:
# High waist circumference indicator
# Male: waist >= 102 cm
# Female: waist >= 88 cm
# RIAGENDR: 1 = male, 2 = female

if "BMXWAIST" in X_eng.columns and "RIAGENDR" in X_eng.columns:
    
    conditions = [
        (X_eng["RIAGENDR"] == 1) & (X_eng["BMXWAIST"] >= 102),
        (X_eng["RIAGENDR"] == 2) & (X_eng["BMXWAIST"] >= 88),
        (X_eng["RIAGENDR"] == 1) & (X_eng["BMXWAIST"] < 102),
        (X_eng["RIAGENDR"] == 2) & (X_eng["BMXWAIST"] < 88)
    ]
    
    choices = [1, 1, 0, 0]
    
    X_eng["high_waist"] = np.select(
        conditions,
        choices,
        default=np.nan
    )

X_eng[["RIAGENDR", "BMXWAIST", "high_waist"]].head()

,RIAGENDR,BMXWAIST,high_waist
0,2.0,117.9,1.0
1,2.0,NaN,NaN
2,1.0,120.4,1.0
3,1.0,86.8,0.0
4,1.0,109.6,1.0


In [50]:
# Physical activity indicator
# physically_active = 1 if any activity variable is Yes
# physically_active = 0 if all available activity variables are No
# physically_active = NaN if all activity variables are missing

activity_cols = [col for col in ["PAQ650", "PAQ665"] if col in X_eng.columns]

if len(activity_cols) > 0:
    X_eng["physically_active"] = np.nan

    # If any activity response is Yes
    X_eng.loc[
        (X_eng[activity_cols] == 1).any(axis=1),
        "physically_active"
    ] = 1

    # If all activity responses are known and all are No
    X_eng.loc[
        X_eng[activity_cols].notna().all(axis=1) &
        (X_eng[activity_cols] != 1).all(axis=1),
        "physically_active"
    ] = 0

X_eng[activity_cols + ["physically_active"]].head()

,PAQ650,PAQ665,physically_active
0,1.0,1.0,1.0
1,1.0,2.0,1.0
2,2.0,2.0,0.0
3,2.0,1.0,1.0
4,2.0,1.0,1.0


In [51]:
# Alcohol use indicator
# ever_regular_alcohol = 1 if ALQ111 == 1
# ever_regular_alcohol = 0 if ALQ111 == 2
# ever_regular_alcohol = NaN if ALQ111 is missing

if "ALQ111" in X_eng.columns:
    X_eng["ever_regular_alcohol"] = np.where(
        X_eng["ALQ111"].isna(),
        np.nan,
        np.where(X_eng["ALQ111"] == 1, 1, 0)
    )

X_eng[["ALQ111", "ever_regular_alcohol"]].head()

,ALQ111,ever_regular_alcohol
0,1.0,1.0
1,NaN,NaN
2,1.0,1.0
3,1.0,1.0
4,1.0,1.0


In [52]:
# Smoking history indicator
# smoking_history = 1 if SMQ020 == 1
# smoking_history = 0 if SMQ020 == 2
# smoking_history = NaN if SMQ020 is missing

if "SMQ020" in X_eng.columns:
    X_eng["smoking_history"] = np.where(
        X_eng["SMQ020"].isna(),
        np.nan,
        np.where(X_eng["SMQ020"] == 1, 1, 0)
    )

X_eng[["SMQ020", "smoking_history"]].head()

,SMQ020,smoking_history
0,2.0,0.0
1,2.0,0.0
2,1.0,1.0
3,1.0,1.0
4,2.0,0.0


In [53]:
# Fair or poor diet indicator
# fair_poor_diet = 1 if DBQ700 is Fair or Poor
# fair_poor_diet = 0 if DBQ700 is Excellent, Very good, or Good
# fair_poor_diet = NaN if DBQ700 is missing

if "DBQ700" in X_eng.columns:
    X_eng["fair_poor_diet"] = np.where(
        X_eng["DBQ700"].isna(),
        np.nan,
        np.where(X_eng["DBQ700"] >= 4, 1, 0)
    )

X_eng[["DBQ700", "fair_poor_diet"]].head()

,DBQ700,fair_poor_diet
0,3.0,0.0
1,1.0,0.0
2,3.0,0.0
3,4.0,1.0
4,2.0,0.0


In [54]:
# Frequent fast-food / restaurant meals indicator
# frequent_fast_food = 1 if DBD895 >= 4
# frequent_fast_food = 0 if DBD895 < 4
# frequent_fast_food = NaN if DBD895 is missing

if "DBD895" in X_eng.columns:
    X_eng["frequent_fast_food"] = np.where(
        X_eng["DBD895"].isna(),
        np.nan,
        np.where(X_eng["DBD895"] >= 4, 1, 0)
    )

X_eng[["DBD895", "frequent_fast_food"]].head()

,DBD895,frequent_fast_food
0,7.0,1.0
1,4.0,1.0
2,2.0,0.0
3,2.0,0.0
4,0.0,0.0


In [55]:
# Interaction terms

if "RIDAGEYR" in X_eng.columns and "BMXBMI" in X_eng.columns:
    X_eng["age_bmi"] = X_eng["RIDAGEYR"] * X_eng["BMXBMI"]

if "RIDAGEYR" in X_eng.columns and "BMXWAIST" in X_eng.columns:
    X_eng["age_waist"] = X_eng["RIDAGEYR"] * X_eng["BMXWAIST"]

if "RIDAGEYR" in X_eng.columns and "avg_systolic_bp" in X_eng.columns:
    X_eng["age_systolic_bp"] = X_eng["RIDAGEYR"] * X_eng["avg_systolic_bp"]

X_eng[["age_bmi", "age_waist", "age_systolic_bp"]].head()

,age_bmi,age_waist,age_systolic_bp
0,1096.2,3419.1,2871.0
1,NaN,NaN,NaN
2,1455.3,5899.6,5243.0
3,788.4,3124.8,4092.0
4,2053.6,7452.8,9112.0


In [56]:
# Simple metabolic risk score

risk_indicator_cols = [
    "obese",
    "high_waist",
    "high_bp_exam",
    "smoking_history",
    "fair_poor_diet"
]

risk_indicator_cols = [col for col in risk_indicator_cols if col in X_eng.columns]

X_eng["metabolic_risk_score"] = X_eng[risk_indicator_cols].sum(axis=1)

X_eng[risk_indicator_cols + ["metabolic_risk_score"]].head()

,obese,high_waist,high_bp_exam,smoking_history,fair_poor_diet,metabolic_risk_score
0,1.0,1.0,0.0,0.0,0.0,2.0
1,NaN,NaN,NaN,0.0,0.0,0.0
2,0.0,1.0,0.0,1.0,0.0,2.0
3,0.0,0.0,0.0,1.0,1.0,2.0
4,1.0,1.0,1.0,0.0,0.0,3.0


In [57]:
print("Shape before feature engineering:", X_pred_selected.shape)
print("Shape after feature engineering:", X_eng.shape)

new_features = [col for col in X_eng.columns if col not in X_pred_selected.columns]

print("New engineered features:")
print(new_features)

Shape before feature engineering: (9232, 23)
Shape after feature engineering: (9232, 37)
New engineered features:
['pulse_pressure', 'obese', 'bmi_category', 'high_bp_exam', 'high_waist', 'physically_active', 'ever_regular_alcohol', 'smoking_history', 'fair_poor_diet', 'frequent_fast_food', 'age_bmi', 'age_waist', 'age_systolic_bp', 'metabolic_risk_score']


In [58]:
X_pred_engineered = X_eng.copy()
print("Final engineered predictor shape:", X_pred_engineered.shape)
X_pred_engineered.head()

Final engineered predictor shape: (9232, 37)


,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPQ020,BPQ080,SMQ020,...,high_waist,physically_active,ever_regular_alcohol,smoking_history,fair_poor_diet,frequent_fast_food,age_bmi,age_waist,age_systolic_bp,metabolic_risk_score
0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,2.0,1.0,2.0,...,1.0,1.0,1.0,0.0,0.0,1.0,1096.2,3419.1,2871.0,2.0
1,21.0,2.0,2.0,4.0,5.00,NaN,NaN,2.0,2.0,2.0,...,NaN,1.0,NaN,0.0,0.0,1.0,NaN,NaN,NaN,0.0
2,49.0,1.0,3.0,2.0,NaN,29.7,120.4,2.0,1.0,1.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1455.3,5899.6,5243.0,2.0
3,36.0,1.0,3.0,4.0,0.83,21.9,86.8,2.0,2.0,1.0,...,0.0,1.0,1.0,1.0,1.0,0.0,788.4,3124.8,4092.0,2.0
4,68.0,1.0,7.0,4.0,1.20,30.2,109.6,1.0,1.0,2.0,...,1.0,1.0,1.0,0.0,0.0,0.0,2053.6,7452.8,9112.0,3.0


In [59]:
# Save engineered predictors

X_pred_engineered.to_csv(
    processed_dir / "X_pred_engineered.csv",
    index=False
)

In [60]:
# Iteration between EDA and feature work
# Compare after and before feature enginnering 

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

In [61]:
coded_categorical_features = [
    # Original coded categorical variables
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "BPQ020",
    "BPQ080",
    "SMQ020",
    "PAQ650",
    "PAQ665",
    "ALQ111",
    "ALQ121",
    "DBQ700",
    
    # Engineered categorical / binary indicator variables
    "obese",
    "bmi_category",
    "high_bp_exam",
    "high_waist",
    "physically_active",
    "ever_regular_alcohol",
    "smoking_history",
    "fair_poor_diet",
    "frequent_fast_food"
]

In [62]:
# Function to create preprocessing pipeline

def build_preprocessor(X):
    categorical_features = [
        col for col in coded_categorical_features 
        if col in X.columns
    ]
    
    object_categorical = X.select_dtypes(include=["object", "category"]).columns.tolist()
    
    categorical_features = list(set(categorical_features + object_categorical))
    
    numeric_features = [
        col for col in X.columns
        if col not in categorical_features
    ]
    
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    
    return preprocessor, numeric_features, categorical_features

In [63]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

In [64]:
# Evaluation matrices 

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score)
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [65]:
# Function to run cross-validation

def evaluate_feature_set(X, y, feature_set_name):
    results = []
    
    preprocessor, numeric_features, categorical_features = build_preprocessor(X)
    
    print(f"\nFeature set: {feature_set_name}")
    print("Number of numeric features:", len(numeric_features))
    print("Number of categorical features:", len(categorical_features))
    
    for model_name, model in models.items():
        
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])
        
        cv_results = cross_validate(
            pipeline,
            X,
            y,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
            n_jobs=-1
        )
        
        row = {
            "feature_set": feature_set_name,
            "model": model_name
        }
        
        for metric in scoring.keys():
            scores = cv_results[f"test_{metric}"]
            row[f"{metric}_mean"] = scores.mean()
            row[f"{metric}_std"] = scores.std()
        
        results.append(row)
    
    return pd.DataFrame(results)

In [68]:
y_target_selected = pd.read_csv(processed_dir / "y_target_selected.csv")

In [69]:
original_results = evaluate_feature_set(
    X_pred_selected,
    y_target_selected,
    "Selected original features"
)

engineered_results = evaluate_feature_set(
    X_pred_engineered,
    y_target_selected,
    "Selected + engineered features"
)

comparison_results = pd.concat(
    [original_results, engineered_results],
    ignore_index=True
)

comparison_results


Feature set: Selected original features
Number of numeric features: 12
Number of categorical features: 11

Feature set: Selected + engineered features
Number of numeric features: 17
Number of categorical features: 20


,feature_set,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std
0,Selected original features,Logistic Regression,0.717723,0.008280,0.733630,0.007163,0.805125,0.006565,0.474789,0.018123,0.759890,0.011733,0.389515,0.008584,0.514966,0.008659
1,Selected original features,Random Forest,0.786289,0.007049,0.689900,0.009681,0.803883,0.003421,0.479604,0.009701,0.530769,0.019223,0.463565,0.014778,0.494726,0.014216
2,Selected + engineered features,Logistic Regression,0.716964,0.006312,0.729219,0.005998,0.806148,0.005983,0.479433,0.016677,0.749451,0.012212,0.387480,0.006497,0.510793,0.006815
3,Selected + engineered features,Random Forest,0.785747,0.005938,0.694744,0.005745,0.802392,0.003946,0.475223,0.011450,0.544505,0.009260,0.463336,0.012176,0.500567,0.009139


In [70]:
# Round results 
comparison_results_rounded = comparison_results.copy()
numeric_result_cols = comparison_results_rounded.select_dtypes(include=["float64"]).columns
comparison_results_rounded[numeric_result_cols] = comparison_results_rounded[numeric_result_cols].round(4)
comparison_results_rounded

,feature_set,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std
0,Selected original features,Logistic Regression,0.7177,0.0083,0.7336,0.0072,0.8051,0.0066,0.4748,0.0181,0.7599,0.0117,0.3895,0.0086,0.5150,0.0087
1,Selected original features,Random Forest,0.7863,0.0070,0.6899,0.0097,0.8039,0.0034,0.4796,0.0097,0.5308,0.0192,0.4636,0.0148,0.4947,0.0142
2,Selected + engineered features,Logistic Regression,0.7170,0.0063,0.7292,0.0060,0.8061,0.0060,0.4794,0.0167,0.7495,0.0122,0.3875,0.0065,0.5108,0.0068
3,Selected + engineered features,Random Forest,0.7857,0.0059,0.6947,0.0057,0.8024,0.0039,0.4752,0.0114,0.5445,0.0093,0.4633,0.0122,0.5006,0.0091


In [71]:
display_cols = [
    "feature_set",
    "model",
    "recall_mean",
    "pr_auc_mean",
    "roc_auc_mean",
    "balanced_accuracy_mean",
    "f1_mean",
    "precision_mean",
    "accuracy_mean"
]
comparison_results_rounded[display_cols]

,feature_set,model,recall_mean,pr_auc_mean,roc_auc_mean,balanced_accuracy_mean,f1_mean,precision_mean,accuracy_mean
0,Selected original features,Logistic Regression,0.7599,0.4748,0.8051,0.7336,0.5150,0.3895,0.7177
1,Selected original features,Random Forest,0.5308,0.4796,0.8039,0.6899,0.4947,0.4636,0.7863
2,Selected + engineered features,Logistic Regression,0.7495,0.4794,0.8061,0.7292,0.5108,0.3875,0.7170
3,Selected + engineered features,Random Forest,0.5445,0.4752,0.8024,0.6947,0.5006,0.4633,0.7857


In [72]:
# Compare engineered features against original selected features

pivot_results = comparison_results_rounded.pivot(
    index="model",
    columns="feature_set",
    values=["recall_mean", "pr_auc_mean", "roc_auc_mean", "balanced_accuracy_mean", "f1_mean"]
)

pivot_results

recall_mean                             \
feature_set         Selected + engineered features Selected original features   
model                                                                           
Logistic Regression                         0.7495                     0.7599   
Random Forest                               0.5445                     0.5308   

                                       pr_auc_mean                             \
feature_set         Selected + engineered features Selected original features   
model                                                                           
Logistic Regression                         0.4794                     0.4748   
Random Forest                               0.4752                     0.4796   

                                      roc_auc_mean                             \
feature_set         Selected + engineered features Selected original features   
model                                                                           
Logistic Regression                         0.8061                     0.8051   
Random Forest                               0.8024                     0.8039   

                            balanced_accuracy_mean                             \
feature_set         Selected + engineered features Selected original features   
model                                                                           
Logistic Regression                         0.7292                     0.7336   
Random Forest                               0.6947                     0.6899   

                                           f1_mean                             
feature_set         Selected + engineered features Selected original features  
model                                                                          
Logistic Regression                         0.5108                     0.5150  
Random Forest                               0.5006                     0.4947